# Tweet Meme vs Real Event Classifier

End-to-end pipeline:

1. **Data Collection** — load PHEME or a CSV with `text` + `timestamp` columns
2. **Preprocessing** — clean text (URLs, mentions, special chars, lowercase)
3. **SBERT Embedding** — `all-mpnet-base-v2`, save `.npy`
4. **BERTopic Clustering** — extract top 5 keywords, peak time, tweet count per cluster
5. **Velocity Calculation** — `tweet_rate_per_hour`, `lifespan_hours`, `peak_to_decay_ratio`
6. **GDELT Labeling** — query ±6 h window, apply lifespan-aware rule, save `labeled_tweets.csv`
7. **Fine-tune BERT** — `bert-base-uncased` on `(tweet_text, label)`, 80/20 split via HF `Trainer`
8. **Evaluation** — classification report + confusion matrix on the test split
9. **Inference** — raw tweet → tokenize → fine-tuned BERT → label + confidence

> Each step saves intermediates to disk so cells can be re-run independently.
> Target: Python 3.10.


## 0. Setup — Install & Import Dependencies

Run once per environment. Pinned versions known-compatible with Python 3.10.


In [1]:
# Uncomment on first run to install dependencies.
!pip install -q "sentence-transformers>=2.7.0" "bertopic>=0.16.0" \
    "transformers>=4.40.0" "datasets>=2.18.0" "accelerate>=0.30.0" \
    "torch>=2.1.0" "scikit-learn>=1.3.0" "pandas>=2.0.0" \
    "numpy>=1.24.0" "requests>=2.31.0" "umap-learn>=0.5.5" "hdbscan>=0.8.33"

import os
import re
import json
import time
import random
import pickle
from pathlib import Path
from datetime import timedelta

import numpy as np
import pandas as pd
import requests

ARTIFACTS = Path("artifacts_meme_vs_event")
ARTIFACTS.mkdir(exist_ok=True)

RAW_CSV          = ARTIFACTS / "raw_tweets.csv"
CLEAN_CSV        = ARTIFACTS / "clean_tweets.csv"
EMBEDDINGS_NPY   = ARTIFACTS / "embeddings.npy"
TOPIC_MODEL_PKL  = ARTIFACTS / "bertopic_model.pkl"
CLUSTERS_CSV     = ARTIFACTS / "clusters.csv"
VELOCITY_CSV     = ARTIFACTS / "velocity.csv"
GDELT_CACHE_JSON = ARTIFACTS / "gdelt_cache.json"
LABELED_CSV      = ARTIFACTS / "labeled_tweets.csv"
BERT_MODEL_DIR   = ARTIFACTS / "bert_classifier"
BERT_TRAINER_DIR = ARTIFACTS / "bert_trainer"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Artifacts directory: {ARTIFACTS.resolve()}")


Artifacts directory: /Users/aryanjangde/dynamic_event_detector/artifacts_meme_vs_event


## 1. Data Collection

Two data sources are supported — toggle each independently with `USE_PHEME`
and `USE_ABCNEWS`:

**PHEME** (`pheme_all.csv`, built by `combine_phmeme.py`):

| column | description |
|---|---|
| `tweet_id` | Twitter id |
| `tweet_text` | raw tweet text |
| `timestamp` | Twitter-style `Tue Mar 24 10:51:21 +0000 2015` |
| `event`, `rumour_type` | metadata |

**ABC News** (`abcnews-date-text.csv`, 1.24M headlines):

| column | description |
|---|---|
| `publish_date` | `YYYYMMDD` integer |
| `headline_text` | headline string |

For ABC News we:

1. draw a deterministic random sample of 200k rows (seeded, cached to `artifacts_meme_vs_event/abcnews_sample.csv` so re-runs are instant),
2. parse `publish_date` as midnight UTC,
3. add a per-row deterministic sub-day offset in seconds so multiple headlines
   from the same day don't collapse onto one timestamp (needed for
   `peak_to_decay_ratio` and `lifespan_hours`).

If neither CSV is available, a small synthetic demo set is used so the
pipeline still runs end-to-end.


In [2]:
# --- data source config -------------------------------------------------
USE_PHEME    = True
USE_ABCNEWS  = True

PHEME_CSV            = "pheme_all.csv"
ABCNEWS_CSV          = "abcnews-date-text.csv"
ABCNEWS_SAMPLE_SIZE  = 200_000
ABCNEWS_SAMPLE_CSV   = ARTIFACTS / "abcnews_sample.csv"  # cached random sample

# Canonical column names used by the rest of the notebook.
TEXT_COL = "tweet_text"
TIME_COL = "timestamp"

# PHEME timestamps look like: "Tue Mar 24 10:51:21 +0000 2015" (Twitter created_at).
PHEME_TS_FORMAT = "%a %b %d %H:%M:%S %z %Y"

# GDELT Doc 2.0 has no coverage before this date — any query before it returns 0.
# Defined here so the ABC News sampler can draw from the eligible window only.
GDELT_COVERAGE_START = pd.Timestamp("2015-02-18", tz="UTC")


def _parse_pheme_timestamp(series: pd.Series) -> pd.Series:
    """Parse Twitter-style PHEME timestamps; fall back to pandas' generic parser."""
    parsed = pd.to_datetime(series, format=PHEME_TS_FORMAT, utc=True, errors="coerce")
    missing = parsed.isna()
    if missing.any():
        parsed.loc[missing] = pd.to_datetime(series[missing], utc=True, errors="coerce")
    return parsed


def _load_pheme(path: str) -> pd.DataFrame:
    """PHEME CSV -> (tweet_text, timestamp)."""
    df = pd.read_csv(path)
    if "tweet_text" not in df.columns or "timestamp" not in df.columns:
        raise ValueError(f"PHEME CSV missing expected columns: {df.columns.tolist()}")
    out = pd.DataFrame({
        TEXT_COL: df["tweet_text"].astype(str),
        TIME_COL: _parse_pheme_timestamp(df["timestamp"]),
    })
    out["source"] = "pheme"
    return out


def _load_abcnews(path: str, sample_size: int, cache: Path) -> pd.DataFrame:
    """Draw a deterministic sample of ABC News headlines from the GDELT-queryable window."""
    # Cache the *sampled* raw rows so re-runs don't have to re-read the 1.24M source CSV.
    if cache.exists():
        sample = pd.read_csv(cache)
    else:
        full = pd.read_csv(path)
        # Sample only from rows GDELT can actually answer for. The full file
        # has ~312k post-2015 rows, which is more than the 200k default target.
        gdelt_start_int = int(GDELT_COVERAGE_START.strftime("%Y%m%d"))
        eligible = full[full["publish_date"] >= gdelt_start_int]
        need = min(sample_size, len(eligible))
        sample = eligible.sample(n=need, random_state=SEED).reset_index(drop=True)
        sample.to_csv(cache, index=False)
        print(
            f"Sampled {len(sample)} ABC News headlines "
            f"from {len(eligible):,} eligible post-{GDELT_COVERAGE_START.date()} rows -> {cache}"
        )

    # publish_date is YYYYMMDD; parse to midnight UTC.
    base_ts = pd.to_datetime(sample["publish_date"].astype(str), format="%Y%m%d", utc=True)

    # Spread multiple headlines from the same day across the day, deterministically,
    # so velocity features (peak_to_decay_ratio, lifespan_hours) have sub-day resolution.
    rng = np.random.default_rng(SEED)
    second_offsets = rng.integers(0, 24 * 3600, size=len(sample))
    ts = base_ts + pd.to_timedelta(second_offsets, unit="s")

    out = pd.DataFrame({
        TEXT_COL: sample["headline_text"].astype(str),
        TIME_COL: ts,
    })
    out["source"] = "abcnews"
    return out


def _synthetic_sample() -> pd.DataFrame:
    """Fallback demo data when no CSV is available."""
    print("No PHEME or ABC News CSV found — using synthetic demo tweets.")
    base = pd.Timestamp("2024-02-06 04:00", tz="UTC")
    demo = []
    for i in range(60):  # earthquake cluster
        demo.append({
            TEXT_COL: random.choice([
                "6.2 magnitude earthquake just hit southern Turkey, buildings shaking",
                "Massive earthquake in Turkey, evacuations underway in Adana",
                "Felt a huge earthquake near the Syrian border, lights flickered",
                "Turkey earthquake: emergency services dispatched to collapse sites",
            ]),
            TIME_COL: base + timedelta(minutes=i * 2),
        })
    for i in range(45):  # AWS outage cluster
        demo.append({
            TEXT_COL: random.choice([
                "AWS us-east-1 is down again, our API is returning 500s",
                "Widespread AWS outage affecting Lambda and S3",
                "us-east-1 region reporting elevated error rates on AWS console",
                "AWS outage is taking half the internet down right now",
            ]),
            TIME_COL: base + timedelta(hours=6) + timedelta(minutes=i * 3),
        })
    for i in range(50):  # meme cluster
        demo.append({
            TEXT_COL: random.choice([
                "skibidi toilet no cap fr fr 💀💀",
                "my sleep schedule is literally skibidi ohio rizz",
                "gyatt level 9000 on this sigma edit",
                "bro really said 'it's giving ohio' 😭",
            ]),
            TIME_COL: base + timedelta(hours=12) + timedelta(minutes=i * 4),
        })
    df = pd.DataFrame(demo)
    df["source"] = "synthetic"
    return df


# --- load selected sources and concatenate ------------------------------
frames: list[pd.DataFrame] = []
if USE_PHEME and Path(PHEME_CSV).exists():
    frames.append(_load_pheme(PHEME_CSV))
    print(f"PHEME: {len(frames[-1])} rows")
if USE_ABCNEWS and Path(ABCNEWS_CSV).exists():
    frames.append(_load_abcnews(ABCNEWS_CSV, ABCNEWS_SAMPLE_SIZE, ABCNEWS_SAMPLE_CSV))
    print(f"ABC News: {len(frames[-1])} rows")

if not frames:
    frames.append(_synthetic_sample())

raw_df = pd.concat(frames, ignore_index=True)
raw_df = raw_df.dropna(subset=[TEXT_COL, TIME_COL]).reset_index(drop=True)

# Safety net: PHEME's 2014 tweets would slip through the ABC News date-floored sampler.
# Keep the corpus strictly within GDELT's coverage window.
before = len(raw_df)
raw_df = raw_df[raw_df[TIME_COL] >= GDELT_COVERAGE_START].reset_index(drop=True)
if before != len(raw_df):
    print(f"Dropped {before - len(raw_df)} rows before {GDELT_COVERAGE_START.date()} (outside GDELT coverage).")

# Deterministic shuffle so PHEME rows aren't all clustered at the front of the DataFrame.
raw_df = raw_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

raw_df.to_csv(RAW_CSV, index=False)
print(f"\nCombined corpus: {len(raw_df)} rows -> {RAW_CSV}")
print(f"Time range: {raw_df[TIME_COL].min()}  →  {raw_df[TIME_COL].max()}")
print("Source mix:")
print(raw_df["source"].value_counts().to_string())
raw_df.head()


PHEME: 6425 rows
ABC News: 200000 rows

Combined corpus: 206425 rows -> artifacts_meme_vs_event/raw_tweets.csv
Time range: 2003-02-19 00:56:41+00:00  →  2021-12-31 16:59:58+00:00
Source mix:
source
abcnews    200000
pheme        6425


,tweet_text,timestamp,source
0,police mourn officers death,2005-04-24 08:10:21+00:00,abcnews
1,salinity played down as fish kill cause,2010-02-18 07:38:20+00:00,abcnews
2,tszyu admits doubts over fight future,2005-10-30 20:49:14+00:00,abcnews
3,timber communities australia backs biomass plant,2009-04-20 11:36:17+00:00,abcnews
4,jobkeeper jobseeker coronavirus supplement cov...,2021-03-10 15:39:15+00:00,abcnews


## 2. Preprocessing

Normalize the raw text so embeddings and keyword extraction aren't dominated by
noise (URLs, mentions, hashtag symbols, emojis, extra whitespace). We keep a
copy of the original text in `raw_text` because inference should show the user's
original string.


In [3]:
_URL_RE      = re.compile(r"https?://\S+|www\.\S+")
_MENTION_RE  = re.compile(r"@\w+")
_HASHTAG_RE  = re.compile(r"#")
_NON_WORD_RE = re.compile(r"[^a-z0-9\s]")
_WS_RE       = re.compile(r"\s+")


def clean_tweet(text: str) -> str:
    """Lowercase, drop URLs/mentions, strip hashtag symbol, remove special chars."""
    if not isinstance(text, str):
        return ""
    t = text.lower()
    t = _URL_RE.sub(" ", t)
    t = _MENTION_RE.sub(" ", t)
    t = _HASHTAG_RE.sub(" ", t)      # keep the word, drop the '#'
    t = _NON_WORD_RE.sub(" ", t)     # drop emojis/punctuation
    t = _WS_RE.sub(" ", t).strip()
    return t


clean_df = raw_df.rename(columns={TEXT_COL: "raw_text"}).copy()
clean_df["clean_text"] = clean_df["raw_text"].map(clean_tweet)

# Drop rows that became empty after cleaning (pure URLs / emojis).
clean_df = clean_df[clean_df["clean_text"].str.len() > 0].reset_index(drop=True)

clean_df.to_csv(CLEAN_CSV, index=False)
print(f"Kept {len(clean_df)} tweets after cleaning -> {CLEAN_CSV}")
clean_df[["raw_text", "clean_text"]].head()


Kept 206425 tweets after cleaning -> artifacts_meme_vs_event/clean_tweets.csv


,raw_text,clean_text
0,police mourn officers death,police mourn officers death
1,salinity played down as fish kill cause,salinity played down as fish kill cause
2,tszyu admits doubts over fight future,tszyu admits doubts over fight future
3,timber communities australia backs biomass plant,timber communities australia backs biomass plant
4,jobkeeper jobseeker coronavirus supplement cov...,jobkeeper jobseeker coronavirus supplement cov...


## 3. SBERT Embedding

Encode each cleaned tweet with `all-mpnet-base-v2` (768-dim). Embeddings are
cached to disk so downstream steps (BERTopic, XGBoost) can be re-run without
re-encoding.


In [4]:
from sentence_transformers import SentenceTransformer

SBERT_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

# Load once and keep in memory for later (inference cell re-uses this).
sbert = SentenceTransformer(SBERT_MODEL_NAME)

# Re-use cached embeddings if available AND they match the row count.
if EMBEDDINGS_NPY.exists():
    cached = np.load(EMBEDDINGS_NPY)
    if cached.shape[0] == len(clean_df):
        embeddings = cached
        print(f"Loaded cached embeddings: {embeddings.shape}")
    else:
        embeddings = None
else:
    embeddings = None

if embeddings is None:
    embeddings = sbert.encode(
        clean_df["clean_text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # cosine-ready
    )
    np.save(EMBEDDINGS_NPY, embeddings)
    print(f"Saved embeddings {embeddings.shape} -> {EMBEDDINGS_NPY}")


/Users/aryanjangde/dynamic_event_detector/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6292.48it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 3226/3226 [05:26<00:00,  9.87it/s]


Saved embeddings (206425, 768) -> artifacts_meme_vs_event/embeddings.npy


## 4. BERTopic Clustering

Run BERTopic on the pre-computed embeddings. We skip BERTopic's internal
embedding step by passing `embedding_model=None` and our own `embeddings=`.

For each cluster (topic id ≥ 0; -1 = outlier/noise) we extract:

- top 5 keywords (via BERTopic's c-TF-IDF)
- peak timestamp (median tweet time inside the cluster)
- tweet count


In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# Scale topic granularity with corpus size:
#   - small corpora (< 1k): tiny clusters of ~5 allowed
#   - medium corpora: 1% floor, capped at 50 so clusters stay interpretable
#   - large corpora (200k+): cap at 200 so we don't collapse into a handful of mega-topics
n_docs = len(clean_df)
if n_docs < 1_000:
    min_topic_size = max(5, n_docs // 50)
elif n_docs < 50_000:
    min_topic_size = max(10, min(n_docs // 100, 50))
else:
    min_topic_size = min(max(n_docs // 500, 50), 200)

# Bigger corpora benefit from a higher min_df to drop noisy one-off tokens.
min_df = 1 if n_docs < 5_000 else 5
vectorizer = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=min_df)

print(f"BERTopic config: n_docs={n_docs}  min_topic_size={min_topic_size}  min_df={min_df}")

topic_model = BERTopic(
    embedding_model=None,          # we pass our own embeddings
    vectorizer_model=vectorizer,
    min_topic_size=min_topic_size,
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(clean_df["clean_text"].tolist(), embeddings)
clean_df["topic"] = topics

# Re-assign outlier (-1) docs to their nearest real topic so they aren't silently
# defaulted to "meme" in step 6. BERTopic's reduce_outliers uses embedding similarity.
outlier_share_before = float((clean_df["topic"] == -1).mean())
if outlier_share_before > 0:
    new_topics = topic_model.reduce_outliers(
        clean_df["clean_text"].tolist(),
        clean_df["topic"].tolist(),
        strategy="embeddings",
        embeddings=embeddings,
    )
    clean_df["topic"] = new_topics
    outlier_share_after = float((clean_df["topic"] == -1).mean())
    print(f"Outlier share: {outlier_share_before:.2%} -> {outlier_share_after:.2%} after reduce_outliers")

# Persist the fitted topic model for re-use (e.g. inference on new tweets).
with open(TOPIC_MODEL_PKL, "wb") as fh:
    pickle.dump(topic_model, fh)


def _temporal_peak(times: pd.Series, bin_hours: int = 6) -> pd.Timestamp:
    """Return the midpoint of the densest `bin_hours` bin in the cluster.

    Using the median of a year-spanning cluster points GDELT at an arbitrary date;
    the densest bin is where the real-world event actually happened.
    """
    if len(times) == 0:
        return pd.NaT
    binned = times.dt.floor(f"{bin_hours}h")
    peak_bin = binned.value_counts().idxmax()
    return peak_bin + pd.Timedelta(hours=bin_hours / 2)


def _cluster_summary(df: pd.DataFrame, model: BERTopic) -> pd.DataFrame:
    """One row per cluster with keywords, temporal peak, and size."""
    rows = []
    for topic_id in sorted(t for t in df["topic"].unique() if t != -1):
        sub = df[df["topic"] == topic_id]
        keywords = [w for w, _ in model.get_topic(topic_id)[:5]]
        rows.append({
            "topic_id": int(topic_id),
            "size": int(len(sub)),
            "peak_timestamp": _temporal_peak(sub[TIME_COL]),
            "keywords": keywords,
        })
    return pd.DataFrame(rows).sort_values("size", ascending=False).reset_index(drop=True)


clusters_df = _cluster_summary(clean_df, topic_model)
# Serialize keywords as JSON so the CSV round-trips cleanly.
clusters_df.assign(keywords=clusters_df["keywords"].map(json.dumps)).to_csv(CLUSTERS_CSV, index=False)

print(f"Found {len(clusters_df)} clusters (excluding noise).")
clusters_df


## 5. Velocity Calculation

Compute three temporal signals per cluster — they help separate fast-spiking
memes from sustained real-world events, and `lifespan_hours` feeds the
GDELT labeling rule in the next step.

- `tweet_rate_per_hour` = `cluster_size / lifespan_hours`
- `lifespan_hours`      = `last_tweet_time − first_tweet_time`
- `peak_to_decay_ratio` = `tweets_in_first_6h / tweets_in_next_18h`
  (high value ⇒ spike-then-die, meme-like)


In [ ]:
def _cluster_velocity_stats(df: pd.DataFrame) -> dict[int, dict]:
    """Compute the three velocity features for each topic id."""
    stats: dict[int, dict] = {}
    for tid, sub in df.groupby("topic"):
        times = sub[TIME_COL].sort_values()
        size = len(times)
        first = times.iloc[0]
        last = times.iloc[-1]
        span_hours = (last - first).total_seconds() / 3600.0

        # Guard against zero-span clusters (all tweets at the same instant).
        safe_span = max(span_hours, 1e-3)
        rate = size / safe_span

        # peak-to-decay: first 6h vs next 18h, anchored at the cluster's first tweet.
        early_cutoff = first + pd.Timedelta(hours=6)
        late_cutoff  = first + pd.Timedelta(hours=24)
        early = int(((times >= first) & (times < early_cutoff)).sum())
        late  = int(((times >= early_cutoff) & (times < late_cutoff)).sum())
        peak_to_decay = early / max(late, 1)  # avoid div-by-zero; high ⇒ spike-then-die

        stats[int(tid)] = {
            "cluster_size": int(size),
            "tweet_rate_per_hour": float(rate),
            "lifespan_hours": float(span_hours),
            "peak_to_decay_ratio": float(peak_to_decay),
        }
    return stats


velocity_stats = _cluster_velocity_stats(clean_df)

# Persist velocity stats for audit / re-use.
velocity_df = (
    pd.DataFrame.from_dict(velocity_stats, orient="index")
    .rename_axis("topic_id")
    .reset_index()
    .sort_values("topic_id")
)
velocity_df.to_csv(VELOCITY_CSV, index=False)

# Convenience lookup: topic_id -> lifespan_hours (used by the GDELT labeling rule next).
lifespan_hours = {tid: s["lifespan_hours"] for tid, s in velocity_stats.items()}

print(f"Computed velocity stats for {len(velocity_df)} clusters -> {VELOCITY_CSV}")
velocity_df


## 6. GDELT Labeling

For every cluster we hit the GDELT DOC 2.0 API with the cluster's top
keywords and a ±6 h window around the peak timestamp, then combine the
match with `lifespan_hours` to decide the label.

**Rule:**

| Condition | Label |
|---|---|
| GDELT match **AND** `lifespan_hours > 12` | `real_event` (1) |
| GDELT match **AND** `lifespan_hours < 6`  | `meme` (0) |
| GDELT match **AND** `6 ≤ lifespan_hours ≤ 12` | `meme` (0) *(ambiguous — safer default)* |
| No GDELT match | `meme` (0) |
| Outlier topic `-1` | `meme` (0) |

- Exponential backoff on `429` / `5xx`
- Results cached to `gdelt_cache.json` so re-runs are free
- Final output: `labeled_tweets.csv` with columns `tweet_text, label`


In [ ]:
GDELT_URL          = "https://api.gdeltproject.org/api/v2/doc/doc"
GDELT_WINDOW_HOURS = 6
GDELT_MIN_ARTICLES = 1           # threshold for "event confirmed"
GDELT_MAX_RETRIES  = 5
GDELT_BASE_DELAY   = 1.5         # seconds
GDELT_TIMEOUT      = 45          # seconds; parenthesized OR queries can be slow


def _gdelt_ts(ts: pd.Timestamp) -> str:
    """GDELT expects timestamps in YYYYMMDDHHMMSS UTC."""
    return ts.tz_convert("UTC").strftime("%Y%m%d%H%M%S")


_SINGLE_WORD_RE = re.compile(r"^[a-zA-Z]\w*$")  # starts with a letter, not pure-numeric
_NUMERIC_RE     = re.compile(r"^\d+$")


def _clean_keyword(kw: str) -> str | None:
    """Drop keywords GDELT will reject: too short, pure numeric, stopword-ish."""
    if not isinstance(kw, str):
        return None
    kw = kw.strip()
    if len(kw) < 3:             # GDELT requires min 3 chars per term
        return None
    if _NUMERIC_RE.match(kw):   # pure numbers get tokenized away
        return None
    return kw


def _format_term(kw: str) -> str:
    """Single word -> bare; multi-word phrase -> GDELT phrase match in quotes."""
    return kw if _SINGLE_WORD_RE.match(kw) else f'"{kw}"'


def _build_query(keywords: list[str]) -> str:
    """Build a GDELT-valid OR query wrapped in parentheses.

    Previous version emitted `a OR b OR c` without parens — GDELT silently
    parses this as AND across all terms, which returned 0 hits for every
    cluster in the cache. The fix:
      1. Drop sub-3-char / pure-numeric keywords (GDELT rejects them).
      2. Wrap OR terms in parentheses — required by the GDELT query parser.
      3. Quote multi-word phrases so "covid 19" becomes a phrase match.
    """
    kws = [c for c in (_clean_keyword(k) for k in keywords) if c][:5]
    if not kws:
        return ""
    if len(kws) == 1:
        return _format_term(kws[0])
    return "(" + " OR ".join(_format_term(k) for k in kws) + ")"


def query_gdelt(keywords: list[str], peak_ts: pd.Timestamp) -> dict:
    """Query GDELT with exponential backoff. Returns {'match': bool, 'article_count': int}."""
    query = _build_query(keywords)
    if not query or pd.isna(peak_ts):
        return {"match": False, "article_count": 0, "error": "empty_query"}

    start = _gdelt_ts(peak_ts - timedelta(hours=GDELT_WINDOW_HOURS))
    end   = _gdelt_ts(peak_ts + timedelta(hours=GDELT_WINDOW_HOURS))

    params = {
        "query": query,
        "mode": "artlist",
        "format": "json",
        "maxrecords": "25",
        "startdatetime": start,
        "enddatetime": end,
    }

    delay = GDELT_BASE_DELAY
    for attempt in range(1, GDELT_MAX_RETRIES + 1):
        try:
            resp = requests.get(GDELT_URL, params=params, timeout=GDELT_TIMEOUT)
            if resp.status_code == 200:
                # GDELT's error responses come back as 200 + plain-text body,
                # e.g. "Queries containing OR'd terms must be surrounded by ().".
                # Detect those by trying to parse JSON — text responses will fail.
                body = resp.text.strip()
                if not body:
                    return {"match": False, "article_count": 0}
                try:
                    data = resp.json()
                except ValueError:
                    # Text-mode error, surface it so the cache isn't silently poisoned.
                    return {"match": False, "article_count": 0, "error": body[:120]}
                articles = data.get("articles", []) if isinstance(data, dict) else []
                count = len(articles)
                return {"match": count >= GDELT_MIN_ARTICLES, "article_count": count}
            if resp.status_code in {429, 500, 502, 503, 504}:
                raise requests.RequestException(f"status {resp.status_code}")
            return {"match": False, "article_count": 0, "error": f"http_{resp.status_code}"}
        except requests.RequestException as exc:
            if attempt == GDELT_MAX_RETRIES:
                return {"match": False, "article_count": 0, "error": str(exc)}
            time.sleep(delay + random.uniform(0, 0.5))  # jitter
            delay *= 2                                  # exponential backoff

    return {"match": False, "article_count": 0, "error": "max_retries"}


# Load cached GDELT responses so re-runs don't re-hit the API.
cache: dict[str, dict] = {}
if GDELT_CACHE_JSON.exists():
    cache = json.loads(GDELT_CACHE_JSON.read_text())

gdelt_results: dict[int, dict] = {}
for _, row in clusters_df.iterrows():
    tid = int(row["topic_id"])
    cache_key = f"{tid}:{row['peak_timestamp']}:{','.join(row['keywords'])}"
    if cache_key in cache:
        gdelt_results[tid] = cache[cache_key]
        continue
    gdelt_results[tid] = query_gdelt(row["keywords"], row["peak_timestamp"])
    cache[cache_key] = gdelt_results[tid]
    time.sleep(0.4)  # polite pacing between clusters

GDELT_CACHE_JSON.write_text(json.dumps(cache, indent=2, default=str))


def _label_for_cluster(topic_id: int) -> int:
    """Apply GDELT + lifespan rule. Outlier (-1) and ambiguous lifespan default to meme."""
    if topic_id == -1:
        return 0
    gdelt = gdelt_results.get(int(topic_id), {}).get("match", False)
    span = lifespan_hours.get(int(topic_id), 0.0)
    if gdelt and span > 12:
        return 1  # real_event: confirmed by news AND sustained
    if gdelt and span < 6:
        return 0  # meme: briefly viral even if some news matched
    return 0      # no match OR ambiguous 6-12h window


# Build the per-tweet label table using the rule above.
labeled_full = clean_df.copy()
labeled_full["label"] = labeled_full["topic"].map(_label_for_cluster).astype(int)
labeled_full["label_name"] = labeled_full["label"].map({0: "meme", 1: "real_event"})

# Per-cluster audit log.
print("Per-cluster decisions (GDELT match, lifespan, label):")
for tid in sorted(t for t in labeled_full["topic"].unique() if t != -1):
    print(
        f"  topic {tid:>2}  size={int((labeled_full['topic'] == tid).sum()):>4}"
        f"  lifespan={lifespan_hours.get(tid, 0):.2f}h"
        f"  gdelt_match={gdelt_results.get(tid, {}).get('match', False)}"
        f"  -> {labeled_full.loc[labeled_full['topic'] == tid, 'label_name'].iloc[0]}"
    )

# Save ONLY the columns the BERT fine-tune step needs.
labeled_out = (
    labeled_full[["raw_text", "label"]]
    .rename(columns={"raw_text": "tweet_text"})
    .reset_index(drop=True)
)
labeled_out.to_csv(LABELED_CSV, index=False)

print("\nLabel distribution:")
print(labeled_out["label"].value_counts().rename({0: "meme", 1: "real_event"}))
print(f"\nSaved -> {LABELED_CSV}  (columns: tweet_text, label)")
labeled_out.head()


## 7. Fine-tune BERT

Fine-tune `bert-base-uncased` on `(tweet_text, label)` from
`labeled_tweets.csv` with an 80/20 stratified split via the HuggingFace
`Trainer` API. The fine-tuned weights + tokenizer are saved to
`artifacts_meme_vs_event/bert_classifier/` so inference can reload without
re-training.

We train on the **raw** tweet text (not the cleaned version) so the model can
learn from signals like hashtags, mentions, and punctuation that the cleaner
strips out.


In [ ]:
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(SEED)

BERT_BASE    = "bert-base-uncased"
MAX_LENGTH   = 128
NUM_EPOCHS   = 3
BATCH_SIZE   = 16
LEARNING_RATE = 2e-5
LABEL_NAMES  = {0: "meme", 1: "real_event"}

# Load from disk so this step can be re-run without the earlier cells.
labeled_df_src = pd.read_csv(LABELED_CSV)
labeled_df_src = labeled_df_src.dropna(subset=["tweet_text", "label"]).reset_index(drop=True)
labeled_df_src["label"] = labeled_df_src["label"].astype(int)

# Stratify if both classes are present, otherwise fall back to a plain random split.
stratify = labeled_df_src["label"] if labeled_df_src["label"].nunique() > 1 else None
train_df, test_df = train_test_split(
    labeled_df_src,
    test_size=0.2,
    random_state=SEED,
    stratify=stratify,
)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)}   Test: {len(test_df)}")
print("Train label counts:", train_df["label"].value_counts().to_dict())
print("Test  label counts:", test_df["label"].value_counts().to_dict())

tokenizer = AutoTokenizer.from_pretrained(BERT_BASE)


def _to_hf_dataset(df: pd.DataFrame) -> Dataset:
    """pandas DataFrame -> tokenized HF Dataset with `input_ids`, `attention_mask`, `labels`."""
    ds = Dataset.from_pandas(df[["tweet_text", "label"]], preserve_index=False)
    ds = ds.map(
        lambda batch: tokenizer(
            batch["tweet_text"],
            truncation=True,
            max_length=MAX_LENGTH,
        ),
        batched=True,
    )
    # Trainer expects the target column to be named `labels`.
    ds = ds.rename_column("label", "labels")
    return ds


train_ds = _to_hf_dataset(train_df)
test_ds  = _to_hf_dataset(test_df)

model = AutoModelForSequenceClassification.from_pretrained(
    BERT_BASE,
    num_labels=2,
    id2label=LABEL_NAMES,
    label2id={v: k for k, v in LABEL_NAMES.items()},
)

training_args = TrainingArguments(
    output_dir=str(BERT_TRAINER_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
)

trainer.train()

# Persist the fine-tuned model + tokenizer so inference can reload them.
trainer.save_model(str(BERT_MODEL_DIR))
tokenizer.save_pretrained(str(BERT_MODEL_DIR))
print(f"\nSaved fine-tuned BERT -> {BERT_MODEL_DIR}")


## 8. Evaluation

Run the fine-tuned model on the held-out test split and print the full
classification report (precision / recall / f1 per class) along with the
confusion matrix.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Trainer.predict returns raw logits; argmax across axis=1 gives the predicted class.
pred_output = trainer.predict(test_ds)
y_true = np.asarray(pred_output.label_ids)
y_pred = np.argmax(pred_output.predictions, axis=1)

print("Classification report:\n")
print(classification_report(
    y_true, y_pred,
    target_names=[LABEL_NAMES[0], LABEL_NAMES[1]],
    zero_division=0,
))
print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))


## 9. Inference

`classify_tweet(text)` takes a raw tweet string and returns:

```python
{"label": "meme" | "real_event", "confidence": float, "probabilities": {...}}
```

Workflow:

1. Reload the fine-tuned BERT + tokenizer from `BERT_MODEL_DIR`
2. Tokenize the raw tweet (same `max_length` as training)
3. Forward pass, apply softmax over logits
4. Return the argmax label + its probability


In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Reload from disk so this cell works even after a kernel restart,
# as long as step 7 has been run at least once.
_infer_tokenizer = AutoTokenizer.from_pretrained(str(BERT_MODEL_DIR))
_infer_model = AutoModelForSequenceClassification.from_pretrained(str(BERT_MODEL_DIR))
_infer_device = "cuda" if torch.cuda.is_available() else "cpu"
_infer_model.to(_infer_device).eval()

_INFER_LABELS = {0: "meme", 1: "real_event"}


def classify_tweet(text: str) -> dict:
    """Single-tweet inference using the fine-tuned BERT classifier."""
    if not isinstance(text, str) or not text.strip():
        return {
            "label": "meme",
            "confidence": 0.0,
            "probabilities": {"meme": 1.0, "real_event": 0.0},
        }

    enc = _infer_tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    ).to(_infer_device)

    with torch.no_grad():
        logits = _infer_model(**enc).logits[0]
    probs = F.softmax(logits, dim=-1).cpu().numpy()

    pred = int(np.argmax(probs))
    return {
        "label": _INFER_LABELS[pred],
        "confidence": float(probs[pred]),
        "probabilities": {
            "meme": float(probs[0]),
            "real_event": float(probs[1]),
        },
    }


# Smoke test on a handful of inputs.
for sample in [
    "Massive 6.5 earthquake just rocked Istanbul, buildings swaying",
    "skibidi toilet ohio rizz level 9000 fr fr 💀",
    "AWS us-east-1 is throwing 500s across the board, APIs degraded",
]:
    print(sample)
    print("  ->", classify_tweet(sample))
    print()
